<a href="https://colab.research.google.com/github/sawantarya232-blip/Deep-Learning/blob/main/Assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("kaustubhb999/tomatoleaf")

Using Colab cache for faster access to the 'tomatoleaf' dataset.


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import kagglehub

print("Downloading dataset using kagglehub...")
# kagglehub automatically downloads and extracts the dataset to a local cache directory
path = kagglehub.dataset_download("kaustubhb999/tomatoleaf")
print("Dataset downloaded to location:", path)

# Inspect the downloaded folder structure
# Usually, images are inside a subfolder like 'tomato' or 'data' inside path
print("\nContents of downloaded path:")
print(os.listdir(path))

# Point to the actual folder containing the leaf class folders
# If there's a subfolder named 'tomato', we attach it to the path
DATASET_DIR = path
if "tomato" in os.listdir(path):
    DATASET_DIR = os.path.join(path, "tomato")

BATCH_SIZE = 32      # Process 32 images at a time
IMG_SIZE = (224, 224) # Resize all images to 224x224 pixels

# Load Training Data (80% of total images)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Load Validation Data (20% of total images)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Extract class names (disease names)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nFound {num_classes} classes:")
for name in class_names:
    print(f" - {name}")

# Speed up data loading using CPU/GPU memory caching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


# Generates random variations of images during training so model learns robust features
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
])


model = models.Sequential([
    # Input dimensions
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    # Augmentation & Pixel Normalization (convert 0-255 pixels to 0.0-1.0 range)
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    # Feature Extraction Block 1 (Detects basic edges and colors)
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Feature Extraction Block 2 (Detects textures and spots)
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Feature Extraction Block 3 (Detects complex spot and blight patterns)
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Feature Extraction Block 4
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Classification Block
    layers.Flatten(),                          # Convert 2D feature maps to 1D vector
    layers.Dense(256, activation='relu'),       # Fully connected layer
    layers.Dropout(0.5),                        # Turn off 50% neurons to prevent overfitting
    layers.Dense(num_classes, activation='softmax') # Output layer (gives class probabilities)
])

# Print model summary
model.summary()


model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Early stopping halts training if validation loss stops decreasing for 3 consecutive epochs
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy over Epochs')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss over Epochs')

plt.show()


def predict_leaf(image_path):
    """Pass an image file path to get the model's prediction."""
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) # Add batch dimension

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = 100 * np.max(tf.nn.softmax(predictions[0]))

    print(f"\nResult: {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")

# Example usage:
# predict_leaf("path_to_test_image.jpg")

Using Colab cache for faster access to the 'tomatoleaf' dataset.
Dataset downloaded to location: /kaggle/input/tomatoleaf

Contents of downloaded path:
['tomato']
Found 11000 files belonging to 2 classes.
Using 8800 files for training.
Found 11000 files belonging to 2 classes.
Using 2200 files for validation.

Found 2 classes:
 - train
 - val


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,664,130 (25.42 MB)

 Trainable params: 6,664,130 (25.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 1294s 5s/step - accuracy: 0.9073 - loss: 0.3238 - val_accuracy: 0.9086 - val_loss: 0.3114
Epoch 2/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 1193s 4s/step - accuracy: 0.9092 - loss: 0.3133 - val_accuracy: 0.9086 - val_loss: 0.3057
Epoch 3/10
268/275 ━━━━━━━━━━━━━━━━━━━━ 28s 4s/step - accuracy: 0.9115 - loss: 0.3128